# BOED Summary Notebook (presentation)

This notebook is a compact overview of the main steps: 
- setup (model, prior, noise, candidates)
- greedy criteria A/D/C/Random and metrics
- alternative selection (QR pivot, Maxvol)
- sensor placement visualization

Tip: reduce N, n_steps, n_budget, n_x_cand, n_t_cand if it is slow.


In [1]:
import numpy as np
import numpy.linalg as la
import matplotlib.pyplot as plt

import sys
from pathlib import Path

def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "boed" / "__init__.py").is_file():
            return p
        if (p / "pyBOED" / "boed" / "__init__.py").is_file():
            return p / "pyBOED"
    return cwd

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for name in list(sys.modules):
    if name == "boed" or name.startswith("boed."):
        del sys.modules[name]

print(f"Using PROJECT_ROOT={PROJECT_ROOT}")
from boed.core import make_u0
from boed.priors.kernels import Gaussian
from boed.priors.gp_priors import GaussianProcessPrior
from boed.pde.advection_diffusion import AdvectionDiffusion1D_CN
from boed.inference import LinearGaussianModel
from boed.design.criteria import DesignCriteria
from boed.design.greedy import run_greedy_oed
from boed.design.selection import compare_to_greedy
from boed.observations.sensors import SpaceTimeSensors
from boed.core.noise import NoiseModel


Using PROJECT_ROOT=/home/mdoumbou/Documents/Biblio_thèse/pyBOED


In [2]:
SEED = 42
rng = np.random.default_rng(SEED)

# Compact settings (fast)
N = 80
dt = 0.01
n_steps = 40
diffusivity = 0.01
velocity = 0.5
noise_sigma = 0.01
n_budget = 6
n_x_cand = 16
n_t_cand = 10

# u0 variants
u0_kinds = ['gaussian', 'double_gaussian', 'sine']

# Prior
kernel = Gaussian(length_scale=0.2, sigma=1.0)
prior = GaussianProcessPrior(kernel, nx=N, mu=None)

# QoI: mean on a sub-interval
qoi_range = (0.25, 0.5)


In [3]:
# Model and operators
model = AdvectionDiffusion1D_CN(N, dt, diffusivity=diffusivity, velocity=velocity)
x_grid = np.linspace(0, 1, N)

# Transition matrix and trajectory operator
M = model.get_transition_matrix()
powers = [np.eye(N)]
for _ in range(n_steps):
    powers.append(M @ powers[-1])
Trajectory_Op = np.vstack(powers)

# QoI
qmin, qmax = qoi_range
mask = (x_grid >= qmin) & (x_grid <= qmax)
if np.sum(mask) == 0:
    raise ValueError('QoI range too small for grid resolution')
L_qoi = np.zeros(N)
L_qoi[mask] = 1.0 / np.sum(mask)

# Candidates
candidates_x = np.linspace(2, N - 3, int(n_x_cand), dtype=int)
candidates_t = np.linspace(0, n_steps, int(n_t_cand), dtype=int)

noise = NoiseModel(sigma_noise=noise_sigma)


## Step 1: Greedy criteria (A/D/C/Random)


In [4]:
print('Running greedy A/D/C designs...')
des_A, hist_A, _ = run_greedy_oed(model, prior.Sigma, noise, candidates_x, candidates_t, n_budget, 'A')
des_D, hist_D, _ = run_greedy_oed(model, prior.Sigma, noise, candidates_x, candidates_t, n_budget, 'D')
des_C, hist_C, _ = run_greedy_oed(model, prior.Sigma, noise, candidates_x, candidates_t, n_budget, 'C', L_qoi=L_qoi)

# Random baseline
rand_idx = rng.choice(len(candidates_x) * len(candidates_t), n_budget, replace=False)
des_rand = []
for idx in rand_idx:
    ti_idx = idx // len(candidates_x)
    xi_idx = idx % len(candidates_x)
    des_rand.append((int(candidates_x[xi_idx]), int(candidates_t[ti_idx])))

criteria = ['A', 'D', 'C', 'Random']
designs = {
    'A': des_A,
    'D': des_D,
    'C': des_C,
    'Random': des_rand,
}


Running greedy A/D/C designs...
--- Greedy OED optimization (criterion A) ---
Step 1/6: x=57, t=40 | Score: 4.9703e+01
Step 2/6: x=32, t=40 | Score: 2.8140e+01
Step 3/6: x=77, t=26 | Score: 9.2361e+00
Step 4/6: x=37, t=35 | Score: 4.0674e+00
Step 5/6: x=77, t=13 | Score: 1.3016e+00
Step 6/6: x=7, t=8 | Score: 3.4264e-01
--- Greedy OED optimization (criterion D) ---
Step 1/6: x=62, t=0 | Score: -1.5702e+03
Step 2/6: x=2, t=0 | Score: -1.5794e+03
Step 3/6: x=32, t=0 | Score: -1.5885e+03
Step 4/6: x=77, t=0 | Score: -1.5972e+03
Step 5/6: x=17, t=0 | Score: -1.6052e+03
Step 6/6: x=47, t=0 | Score: -1.6126e+03
--- Greedy OED optimization (criterion C) ---
Step 1/6: x=42, t=31 | Score: 2.1890e-04
Step 2/6: x=37, t=17 | Score: 1.1999e-04
Step 3/6: x=42, t=35 | Score: 6.2827e-05
Step 4/6: x=42, t=26 | Score: 5.1324e-05
Step 5/6: x=7, t=8 | Score: 3.4413e-05
Step 6/6: x=37, t=26 | Score: 2.8190e-05


In [6]:
# Evaluation helper
results = {}

def evaluate_design(u0, design, noise_vec):
    unique_pts = sorted(list(set(design)), key=lambda x: x[1])
    sensors = SpaceTimeSensors([p[0] for p in unique_pts], [p[1] for p in unique_pts], N)
    W = sensors.observation_operator(n_steps + 1)
    A_fwd = W @ Trajectory_Op

    y_obs = A_fwd @ u0 + noise_sigma * noise_vec[: len(unique_pts)]
    Sigma_eps = (noise_sigma ** 2) * np.eye(len(unique_pts))

    lgm = LinearGaussianModel(model=model,n_steps=N,Sigma_obs= Sigma_eps, mu_prior=prior.mu, Sigma_prior=prior.Sigma)
    mu_post, Sigma_post = lgm.posterior(y_obs)

    eig = DesignCriteria.EIG(Sigma_post, prior.Sigma)
    err_global = la.norm(u0 - mu_post)
    err_qoi = abs(L_qoi @ u0 - L_qoi @ mu_post)

    return mu_post, Sigma_post, sensors, eig, err_global, err_qoi

for kind in u0_kinds:
    u0 = make_u0(x_grid, kind)
    noise_vec = rng.standard_normal(size=len(candidates_x) * len(candidates_t))
    for crit in criteria:
        mu, sig, sens, eig, err_g, err_q = evaluate_design(u0, designs[crit], noise_vec)
        results[(kind, crit)] = {
            'u0': u0,
            'mu_post': mu,
            'Sigma_post': sig,
            'sensors': sens,
            'eig': eig,
            'err_global': err_g,
            'err_qoi': err_q,
        }

print('Greedy evaluation done.')


ValueError: Sigma_obs must have shape (80, 80), got (6, 6).

In [ ]:
# Heatmaps
def metric_matrix(metric):
    mat = np.zeros((len(u0_kinds), len(criteria)))
    for i, kind in enumerate(u0_kinds):
        for j, crit in enumerate(criteria):
            mat[i, j] = results[(kind, crit)][metric]
    return mat

def plot_heatmap(mat, title, cmap='viridis', fmt='{:.2f}'):
    fig, ax = plt.subplots(figsize=(7, 4))
    im = ax.imshow(mat, cmap=cmap, aspect='auto')
    ax.set_title(title)
    ax.set_xticks(range(len(criteria)))
    ax.set_xticklabels(criteria)
    ax.set_yticks(range(len(u0_kinds)))
    ax.set_yticklabels(u0_kinds)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, fmt.format(mat[i, j]), ha='center', va='center', color='white')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

plot_heatmap(metric_matrix('eig'), 'EIG (nats)')
plot_heatmap(metric_matrix('err_global'), 'Global error ||u0 - mu||', cmap='magma')
plot_heatmap(metric_matrix('err_qoi'), 'QoI error', cmap='magma')


## Step 2: Alternative selection (QR pivot, Maxvol)

Plots are in the Interactive section below (to avoid duplicate figures when parameters change).


## Interactive (optional)

Explore parameters and compare multiple u0 kinds.
Enable save_figs to export plots to results/summary/.


In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display

from boed.priors.kernels import Gaussian, Matern32, Matern52


In [ ]:
def run_summary_interactive(
    N, n_steps, n_budget, n_x_cand, n_t_cand, noise_sigma,
    prior_choice, length_scale, prior_sigma,
    u0_kinds, qoi_min, qoi_max, seed, show_recon, save_figs
):
    if len(u0_kinds) == 0:
        print('Select at least one u0 kind.')
        return

    rng = np.random.default_rng(int(seed))
    dt = 0.01
    diffusivity = 0.01
    velocity = 0.5

    out_dir = os.path.join('results', 'summary')
    if save_figs:
        os.makedirs(out_dir, exist_ok=True)

    def savefig(fig, name):
        if save_figs:
            fig.savefig(os.path.join(out_dir, name), dpi=150)

    model = AdvectionDiffusion1D_CN(N, dt, diffusivity=diffusivity, velocity=velocity)
    x_grid = np.linspace(0, 1, N)

    # Prior
    if prior_choice == 'SE':
        kernel = Gaussian(length_scale=length_scale, sigma=prior_sigma)
    elif prior_choice == 'Matern32':
        kernel = Matern32(length_scale=length_scale, sigma=prior_sigma)
    elif prior_choice == 'Matern52':
        kernel = Matern52(length_scale=length_scale, sigma=prior_sigma)
    else:
        raise ValueError('Unknown prior choice')
    prior = GaussianProcessPrior(kernel, nx=N)

    # QoI
    qmin = min(qoi_min, qoi_max)
    qmax = max(qoi_min, qoi_max)
    mask = (x_grid >= qmin) & (x_grid <= qmax)
    if np.sum(mask) == 0:
        raise ValueError('QoI range too small for grid resolution')
    L_qoi = np.zeros(N)
    L_qoi[mask] = 1.0 / np.sum(mask)

    # Candidates
    candidates_x = np.linspace(2, N - 3, int(n_x_cand), dtype=int)
    candidates_t = np.linspace(0, n_steps, int(n_t_cand), dtype=int)
    noise = NoiseModel(sigma_noise=noise_sigma)

    # Trajectory operator
    M = model.get_transition_matrix()
    powers = [np.eye(N)]
    for _ in range(n_steps):
        powers.append(M @ powers[-1])
    Trajectory_Op = np.vstack(powers)

    # Greedy designs
    des_A, _, _ = run_greedy_oed(model, prior.Sigma, noise, candidates_x, candidates_t, n_budget, 'A')
    des_D, _, _ = run_greedy_oed(model, prior.Sigma, noise, candidates_x, candidates_t, n_budget, 'D')
    des_C, _, _ = run_greedy_oed(model, prior.Sigma, noise, candidates_x, candidates_t, n_budget, 'C', L_qoi=L_qoi)

    # Random baseline
    rand_idx = rng.choice(len(candidates_x) * len(candidates_t), n_budget, replace=False)
    des_rand = []
    for idx in rand_idx:
        ti_idx = idx // len(candidates_x)
        xi_idx = idx % len(candidates_x)
        des_rand.append((int(candidates_x[xi_idx]), int(candidates_t[ti_idx])))

    criteria = ['A', 'D', 'C', 'Random']
    designs = {
        'A': des_A,
        'D': des_D,
        'C': des_C,
        'Random': des_rand,
    }

    # Evaluate selected u0 kinds
    results = {}
    for kind in u0_kinds:
        u0 = make_u0(x_grid, kind)
        noise_vec = rng.standard_normal(size=len(candidates_x) * len(candidates_t))

        def evaluate_design(design):
            unique_pts = sorted(list(set(design)), key=lambda x: x[1])
            sensors = SpaceTimeSensors([p[0] for p in unique_pts], [p[1] for p in unique_pts], N)
            W = sensors.observation_operator(n_steps + 1)
            A_fwd = W @ Trajectory_Op

            y_obs = A_fwd @ u0 + noise_sigma * noise_vec[: len(unique_pts)]
            Sigma_eps = (noise_sigma ** 2) * np.eye(len(unique_pts))

            lgm = LinearGaussianModel(A_fwd, Sigma_eps, prior.mu, prior.Sigma)
            mu_post, Sigma_post = lgm.posterior(y_obs)

            eig = DesignCriteria.EIG(Sigma_post, prior.Sigma)
            err_global = la.norm(u0 - mu_post)
            err_qoi = abs(L_qoi @ u0 - L_qoi @ mu_post)

            return mu_post, Sigma_post, sensors, eig, err_global, err_qoi

        for name, design in designs.items():
            mu_post, Sigma_post, sensors, eig, err_g, err_q = evaluate_design(design)
            results[(kind, name)] = {
                'u0': u0,
                'mu_post': mu_post,
                'Sigma_post': Sigma_post,
                'sensors': sensors,
                'eig': eig,
                'err_global': err_g,
                'err_qoi': err_q,
            }

    # Metrics visualization
    if len(u0_kinds) > 1:
        def metric_matrix(metric):
            mat = np.zeros((len(u0_kinds), len(criteria)))
            for i, kind in enumerate(u0_kinds):
                for j, crit in enumerate(criteria):
                    mat[i, j] = results[(kind, crit)][metric]
            return mat

        def plot_heatmap(mat, title, cmap='viridis', fmt='{:.2f}', fname=None):
            fig, ax = plt.subplots(figsize=(7, 4))
            im = ax.imshow(mat, cmap=cmap, aspect='auto')
            ax.set_title(title)
            ax.set_xticks(range(len(criteria)))
            ax.set_xticklabels(criteria)
            ax.set_yticks(range(len(u0_kinds)))
            ax.set_yticklabels(u0_kinds)
            for i in range(mat.shape[0]):
                for j in range(mat.shape[1]):
                    ax.text(j, i, fmt.format(mat[i, j]), ha='center', va='center', color='white')
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            plt.tight_layout()
            if fname:
                savefig(fig, fname)
            plt.show()

        plot_heatmap(metric_matrix('eig'), 'EIG (nats)', fname='heatmap_eig.png')
        plot_heatmap(metric_matrix('err_global'), 'Global error', cmap='magma', fname='heatmap_err_global.png')
        plot_heatmap(metric_matrix('err_qoi'), 'QoI error', cmap='magma', fname='heatmap_err_qoi.png')
    else:
        kind = u0_kinds[0]
        labels = criteria
        eig_vals = [results[(kind, k)]['eig'] for k in labels]
        err_g_vals = [results[(kind, k)]['err_global'] for k in labels]
        err_q_vals = [results[(kind, k)]['err_qoi'] for k in labels]

        fig, axes = plt.subplots(1, 3, figsize=(12, 3))
        axes[0].bar(labels, eig_vals)
        axes[0].set_title('EIG (nats)')
        axes[1].bar(labels, err_g_vals)
        axes[1].set_title('Global error')
        axes[2].bar(labels, err_q_vals)
        axes[2].set_title('QoI error')
        for ax in axes:
            ax.grid(alpha=0.3)
        plt.tight_layout()
        savefig(fig, 'metrics_bars.png')
        plt.show()

    # Reconstructions
    if show_recon:
        for kind in u0_kinds:
            fig, axes = plt.subplots(1, len(criteria), figsize=(4 * len(criteria), 3))
            if len(criteria) == 1:
                axes = [axes]
            for ax, crit in zip(axes, criteria):
                u0 = results[(kind, crit)]['u0']
                mu_post = results[(kind, crit)]['mu_post']
                Sigma_post = results[(kind, crit)]['Sigma_post']
                std_post = np.sqrt(np.diag(Sigma_post))

                ax.plot(x_grid, u0, lw=2, label='True u0')
                ax.plot(x_grid, mu_post, lw=2, ls='--', label='Posterior')
                ax.fill_between(x_grid, mu_post - 2 * std_post, mu_post + 2 * std_post, alpha=0.2)
                ax.set_title(f'{kind} | {crit}')
                ax.grid(alpha=0.3)
                if ax is axes[0]:
                    ax.legend()

            plt.tight_layout()
            savefig(fig, f'recon_{kind}.png')
            plt.show()

    # Alternative selection vs greedy
    res_alt = compare_to_greedy(
        model=model,
        Sigma_prior=prior.Sigma,
        noise_model=noise,
        candidates_x=candidates_x,
        candidates_t=candidates_t,
        n_budget=n_budget,
        criterion_type='A',
    )

    fig = plt.figure(figsize=(6, 3))
    for label, key, style in [
        ('Greedy', 'greedy', 'o-'),
        ('QR pivot', 'qr', 's-'),
        ('Maxvol', 'maxvol', '^-'),
    ]:
        history = res_alt[key]['history']
        steps = np.arange(1, len(history) + 1)
        plt.plot(steps, history, style, label=label)
    plt.xlabel('Budget (k)')
    plt.ylabel('A-opt score')
    plt.title('Greedy vs QR vs Maxvol')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    savefig(fig, 'aopt_history.png')
    plt.show()

    # Sensor placement (Greedy/QR/Maxvol)
    u0_ref = make_u0(x_grid, u0_kinds[0])
    trajectory = model.evolve(u0_ref, n_steps)

    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, name in zip(axes, ['greedy', 'qr', 'maxvol']):
        design = res_alt[name]['design']
        sensors = SpaceTimeSensors([p[0] for p in design], [p[1] for p in design], N)
        ax.imshow(trajectory.T, aspect='auto', origin='lower', extent=[0, N, 0, n_steps], cmap='viridis')
        ax.scatter(sensors.x_idx, sensors.t_idx, c='r', marker='x', s=80, linewidth=2)
        ax.set_title(name.upper())
        ax.set_xlabel('Space')
        ax.set_ylabel('Time')

    plt.tight_layout()
    savefig(fig, 'sensor_placement.png')
    plt.show()


In [ ]:
N_w = widgets.IntSlider(value=80, min=40, max=160, step=10, description='N', continuous_update=False)
n_steps_w = widgets.IntSlider(value=40, min=10, max=120, step=10, description='n_steps', continuous_update=False)
n_budget_w = widgets.IntSlider(value=6, min=2, max=12, step=1, description='budget', continuous_update=False)
n_x_cand_w = widgets.IntSlider(value=16, min=8, max=30, step=2, description='cand_x', continuous_update=False)
n_t_cand_w = widgets.IntSlider(value=10, min=6, max=20, step=2, description='cand_t', continuous_update=False)
noise_sigma_w = widgets.FloatLogSlider(value=1e-2, base=10, min=-4, max=-1, step=0.1, description='noise', continuous_update=False)

prior_choice_w = widgets.Dropdown(options=['SE', 'Matern32', 'Matern52'], value='SE', description='prior')
length_w = widgets.FloatSlider(value=0.2, min=0.05, max=1.0, step=0.05, description='length', continuous_update=False)
prior_sigma_w = widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description='sigma', continuous_update=False)

u0_kind_w = widgets.SelectMultiple(
    options=['gaussian', 'double_gaussian', 'sine', 'box'],
    value=('gaussian',),
    description='u0',
)
qoi_min_w = widgets.FloatSlider(value=0.25, min=0.0, max=0.9, step=0.05, description='qoi_min', continuous_update=False)
qoi_max_w = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description='qoi_max', continuous_update=False)
seed_w = widgets.IntSlider(value=42, min=0, max=999, step=1, description='seed', continuous_update=False)
show_recon_w = widgets.Checkbox(value=True, description='show recon')
save_figs_w = widgets.Checkbox(value=False, description='save_figs')

ui = widgets.VBox([
    widgets.HBox([N_w, n_steps_w, n_budget_w]),
    widgets.HBox([n_x_cand_w, n_t_cand_w, noise_sigma_w]),
    widgets.HBox([prior_choice_w, length_w, prior_sigma_w]),
    widgets.HBox([u0_kind_w, qoi_min_w, qoi_max_w, seed_w]),
    widgets.HBox([show_recon_w, save_figs_w]),
])

out = widgets.interactive_output(
    run_summary_interactive,
    {
        'N': N_w,
        'n_steps': n_steps_w,
        'n_budget': n_budget_w,
        'n_x_cand': n_x_cand_w,
        'n_t_cand': n_t_cand_w,
        'noise_sigma': noise_sigma_w,
        'prior_choice': prior_choice_w,
        'length_scale': length_w,
        'prior_sigma': prior_sigma_w,
        'u0_kinds': u0_kind_w,
        'qoi_min': qoi_min_w,
        'qoi_max': qoi_max_w,
        'seed': seed_w,
        'show_recon': show_recon_w,
        'save_figs': save_figs_w,
    },
)

display(ui, out)
